In [ ]:
import pandas as pd
import numpy as np
import os
import glob

In [ ]:
TRACKING_FOLDER = r"C:\AAKASH\MS_NOTES\THESIS\Material\Data\recordings\sequence_model\extracted_features_3"
OUTPUT_FOLDER   = r"C:\AAKASH\MS_NOTES\THESIS\Material\Data\recordings\sequence_model\data"

N_SEQUENCES     = 6       # divide each video into 6 equal sequences
N_FEATURES      = 25      # 2 ball + 22 rod positions + 1 frame_norm = 25 total

# Labels: 0=White, 1=Black, 2=Draw
LABELS = {
    'video12_1':  2, 'video12_2':  0, 'video12_3':  0,
    'video12_4':  2, 'video12_5':  2, 'video12_6':  0,
    'video12_7':  1, 'video12_8':  0, 'video12_9':  1,
    'video12_10': 0, 'video12_11': 0,
    'video14_1':  0, 'video14_2':  0, 'video14_3':  1,
    'video14_4':  0, 'video14_5':  0,
    'video15_1':  0, 'video15_2':  0, 'video15_3':  0,
    'video15_4':  2, 'video15_5':  0, 'video15_6':  2,
    'video43_1':  0, 'video43_2':  0, 'video43_3':  2,
    'video43_4':  0, 'video43_5':  0,
    'video45_1':  0, 'video45_2':  2, 'video45_3':  2,
    'video45_4':  0, 'video45_5':  0, 'video45_6':  1,
    'video45_7':  2, 'video45_8':  2, 'video45_9':  2,
    'video45_10': 2,
    'video56_1':  2, 'video56_2':  1, 'video56_3':  1,
    'video56_4':  0, 'video56_5':  0, 'video56_6':  1,
    'video56_7':  2,
    'video61_1':  2, 'video61_2':  0, 'video61_3':  0,
    'video61_4':  1, 'video61_5':  0,
    'video64_1':  1, 'video64_2':  0, 'video64_3':  0,
    'video64_4':  0, 'video64_5':  1, 'video64_6':  1,
    'video64_7':  2, 'video64_8':  0,
}

# Test videos — held out by video, not by sequence
# 3 Black, 3 White, 2 Draw
TEST_VIDEOS = [
    'video12_7',   # Black
    'video56_6',   # Black
    'video14_3',   # Black
    'video12_10',  # White
    'video14_5',   # White
    'video15_5',   # White
    'video12_4',   # Draw
    'video43_3',   # Draw
]

TRAIN_VIDEOS = [v for v in LABELS if v not in TEST_VIDEOS]

# 24 CSV feature columns (2 ball + 22 rod positions)
# frame_norm is not in this list — recomputed per sequence in the loop below
FEATURE_COLS = [
    'x_ball',
    'y_ball',
    'y_rod_1_p1',
    'y_rod_2_p1', 'y_rod_2_p2',
    'y_rod_3_p1', 'y_rod_3_p2', 'y_rod_3_p3',
    'y_rod_4_p1', 'y_rod_4_p2', 'y_rod_4_p3', 'y_rod_4_p4', 'y_rod_4_p5',
    'y_rod_5_p1', 'y_rod_5_p2', 'y_rod_5_p3', 'y_rod_5_p4', 'y_rod_5_p5',
    'y_rod_6_p1', 'y_rod_6_p2', 'y_rod_6_p3',
    'y_rod_7_p1', 'y_rod_7_p2',
    'y_rod_8_p1',
]
# frame_norm added as 25th feature inside the loop below
# Total features = 24 from CSV + 1 frame_norm recomputed = 25

In [ ]:
def process_video_to_sequences(csv_path, video_name):
    df = pd.read_csv(csv_path)

    # Keep only On Field and Occluded rows
    df = df[df['Status'].isin(['On Field', 'Occluded'])].reset_index(drop=True)

    n_clean = len(df)
    print(f"  {video_name}: {n_clean} clean frames after dropping Goal/Wall/Searching")

    if n_clean < N_SEQUENCES:
        print(f"  WARNING: Not enough clean frames. Skipping.")
        return None

    # Divide into 6 equal sequences
    seq_len   = n_clean // N_SEQUENCES      # e.g. 1380 // 6 = 230
    remainder = n_clean  % N_SEQUENCES      # leftover frames i.e dropped (max 5 frames)

    sequences = []

    for i in range(N_SEQUENCES):
        start  = i * seq_len
        end    = start + seq_len
        seq_df = df.iloc[start:end].copy().reset_index(drop=True)

        # Recompute frame_norm within this sequence
        n = len(seq_df)
        seq_df['frame_norm'] = np.arange(n) / (n - 1) if n > 1 else np.zeros(n)

        # Extract 24 CSV features + frame_norm = 25 features
        feature_data = seq_df[FEATURE_COLS].values                    # (seq_len, 24)
        frame_norm   = seq_df['frame_norm'].values.reshape(-1, 1)     # (seq_len, 1)
        seq_array    = np.hstack([feature_data, frame_norm])          # (seq_len, 25)

        sequences.append(seq_array.astype(np.float32))

    print(f"  -> {N_SEQUENCES} sequences x {seq_len} frames x 25 features")
    return sequences

In [ ]:
#Processes all videos in video_list.
def build_dataset(video_list, folder):
    
    X, y, names = [], [], []

    for video in video_list:
        csv_path = os.path.join(folder, f"{video}_tracking.csv")

        if not os.path.exists(csv_path):
            print(f"  MISSING: {csv_path}")
            continue

        sequences = process_video_to_sequences(csv_path, video)

        if sequences is None:
            continue

        label = LABELS[video]

        for i, seq in enumerate(sequences):
            X.append(seq)
            y.append(label)
            names.append(f"{video}_seq{i+1}")

    return X, y, names

In [ ]:
#Pads all sequences to the same length (max length in dataset).
def pad_sequences(X):
    max_len = max(seq.shape[0] for seq in X)
    n_feat  = X[0].shape[1]

    X_padded = np.zeros((len(X), max_len, n_feat), dtype=np.float32)

    for i, seq in enumerate(X):
        X_padded[i, :len(seq), :] = seq

    return X_padded, max_len

In [ ]:
print(f" \n Building TRAIN dataset ({len(TRAIN_VIDEOS)} videos -> {len(TRAIN_VIDEOS)*6} sequences)")

X_train_raw, y_train, train_names = build_dataset(TRAIN_VIDEOS, TRACKING_FOLDER)

print(f" \n Building TEST dataset ({len(TEST_VIDEOS)} videos -> {len(TEST_VIDEOS)*6} sequences)")

X_test_raw, y_test, test_names = build_dataset(TEST_VIDEOS, TRACKING_FOLDER)

#Pad to same length
# Use train max_len as reference and pad test to same length
X_train, max_len = pad_sequences(X_train_raw)
X_test,  _       = pad_sequences(X_test_raw)

# Ensure test sequence length matches train sequence length
if X_test.shape[1] < max_len:
    pad    = np.zeros((X_test.shape[0], max_len - X_test.shape[1], X_train.shape[2]), dtype=np.float32)
    X_test = np.concatenate([X_test, pad], axis=1)
elif X_test.shape[1] > max_len:
    X_test = X_test[:, :max_len, :]

y_train = np.array(y_train, dtype=np.int64)
y_test  = np.array(y_test,  dtype=np.int64)

# Print summary 
print(f" \n DATASET SUMMARY")

print(f"X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}    y_test:  {y_test.shape}")
print(f"Max sequence length: {max_len} frames")
print(f"\nTrain label distribution:")
print(f"  White (0): {(y_train==0).sum()} sequences")
print(f"  Black (1): {(y_train==1).sum()} sequences")
print(f"  Draw  (2): {(y_train==2).sum()} sequences")
print(f"\nTest label distribution:")
print(f"  White (0): {(y_test==0).sum()} sequences")
print(f"  Black (1): {(y_test==1).sum()} sequences")
print(f"  Draw  (2): {(y_test==2).sum()} sequences")

#Save
np.save(os.path.join(OUTPUT_FOLDER, 'X_train.npy'), X_train)
np.save(os.path.join(OUTPUT_FOLDER, 'y_train.npy'), y_train)
np.save(os.path.join(OUTPUT_FOLDER, 'X_test.npy'),  X_test)
np.save(os.path.join(OUTPUT_FOLDER, 'y_test.npy'),  y_test)

print(f"\nSaved: X_train.npy, y_train.npy, X_test.npy, y_test.npy")
print(f"To: {OUTPUT_FOLDER}")

# Save sequence names for debugging
with open(os.path.join(OUTPUT_FOLDER, 'train_sequence_names.txt'), 'w') as f:
    for name, label in zip(train_names, y_train):
        f.write(f"{name}  label={label}\n")

with open(os.path.join(OUTPUT_FOLDER, 'test_sequence_names.txt'), 'w') as f:
    for name, label in zip(test_names, y_test):
        f.write(f"{name}  label={label}\n")

print(f"Saved: train_sequence_names.txt, test_sequence_names.txt")